In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [3]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
label2id = {str(k): int(v) for k, v in model.config.label2id.items()}
num_labels = model.config.num_labels

clf_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    function_to_apply="none",
    top_k=None,
)

print(model_name)
print("id2label:", id2label)
print("label2id:", label2id)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1}


In [4]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
def stable_softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def label_to_index(label):
    label = str(label)
    if label in label2id:
        return int(label2id[label])
    if label.startswith("LABEL_"):
        return int(label.split("_")[-1])
    if label.isdigit():
        return int(label)
    raise KeyError(f"Unknown label from pipeline output: {label}")

batch_size = 64
raw_logits = np.zeros((len(inputs), num_labels), dtype=np.float32)

for start in tqdm(range(0, len(inputs), batch_size)):
    batch_inputs = inputs[start:start + batch_size]
    batch_outputs = clf_pipe(
        batch_inputs,
        batch_size=batch_size,
        truncation=True,
        max_length=128,
    )

    for row_offset, example_outputs in enumerate(batch_outputs):
        row_idx = start + row_offset
        for item in example_outputs:
            cls_idx = label_to_index(item["label"])
            raw_logits[row_idx, cls_idx] = float(item["score"])

probs = stable_softmax(raw_logits, axis=-1)
y_pred = np.argmax(probs, axis=-1)
positive_probs = probs[:, 1]

print("done")
print("raw_logits_shape:", raw_logits.shape)
print("probs_shape:", probs.shape)

  0%|          | 0/7 [00:00<?, ?it/s]

done
raw_logits_shape: (408, 2)
probs_shape: (408, 2)


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
target_names = [id2label[i] for i in range(num_labels)]

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=target_names))

{'accuracy': 0.8578431372549019, 'f1': 0.9026845637583892}
              precision    recall  f1-score   support

     LABEL_0       0.89      0.63      0.74       129
     LABEL_1       0.85      0.96      0.90       279

    accuracy                           0.86       408
   macro avg       0.87      0.80      0.82       408
weighted avg       0.86      0.86      0.85       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])])
    print("raw_logits:", raw_logits[i].tolist())
    print("prob_positive:", float(positive_probs[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: LABEL_1
raw_logits: [-2.142564296722412, 1.9460961818695068]
prob_positive: 0.983514666557312
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: LABEL_0
raw_logits: [0.8126692771911621, -0.6876999139785767]
prob_positive: 0.18237046897411346
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 label: LABEL_0
raw_logi

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("raw_logits:", raw_logits[i].tolist())
    print("prob_positive:", float(positive_probs[i]))

num_errors: 58
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1
raw_logits: [-1.3045603036880493, 1.2749905586242676]
prob_positive: 0.929533839225769
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1
raw_logits: [-0.8472589254379272, 0.8298810720443726]
prob_positive: 0.8425254225730896
idx: 35
sentence1: Bush wanted " to see an aircraft landing the same way that the pilots saw an aircraft landing , " White House press secretary Ari Fleischer said yesterday .
sentence2: On Tuesday , before Byrd 's speech , Fleischer said Bush wanted ' ' to see

In [9]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "scoring_mode": "raw_logits_manual_softmax",
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'scoring_mode': 'raw_logits_manual_softmax',
 'num_examples': 408,
 'accuracy': 0.8578431372549019,
 'f1': 0.9026845637583892}